[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20cleaning/practice/data_cleaning_worksheet.ipynb)

# Practice · data cleaning

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Ten questions on one file. Each question names a variable. Put your result in that variable and run
the cell. The worked answer sits under **Answer**. Click it open once you have tried.

**The data.** 620 outpatient visits, generated for this worksheet. The defects are planted:
sentinel strings, mixed units, mixed date formats, repeated rows. Nothing has been cleaned.

Columns: `visit_id`, `patient_ref`, `visit_date`, `department`, `age`, `weight_kg`, `systolic_bp`,
`phone`.

Read the `dtypes` before you start.


In [ ]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20cleaning/practice/data/"
visits = pd.read_csv(URL + "clinic_visits.csv")

print(visits.shape)
print(visits.dtypes)
visits.head()

### Q1  Sentinel values in systolic_bp

`systolic_bp` carries blanks and three sentinel strings: `-`, `999` and `not recorded`.
Report what `isna()` counts on that column, and the count once the sentinels are included.

Answer variable `q1`: a tuple `(isna_count, true_count)`.

In [ ]:
q1 = ...   # your answer
q1

<details>
<summary><b>Answer</b></summary>

```python
SENTINELS = ["-", "999", "not recorded"]

q1 = (int(visits["systolic_bp"].isna().sum()),
      int((visits["systolic_bp"].isna() | visits["systolic_bp"].isin(SENTINELS)).sum()))
q1
```

```
(28, 62)
```

</details>

### Q2  Repairing the dtype

Those strings are why `systolic_bp` reads as `object`. Convert it to numeric with the
sentinels turned into `NaN`, then report its mean rounded to 1 decimal.

Answer variable `q2`: a `float`.

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
bp = pd.to_numeric(visits["systolic_bp"], errors="coerce").mask(lambda s: s == 999)
q2 = float(bp.mean().round(1))
q2
```

```
128.7
```

</details>

### Q3  Exact duplicates and repeated ids

Count the rows that repeat whole, and the rows that repeat a `visit_id`. The two counts differ.

Answer variable `q3`: a tuple `(exact, by_visit_id)`.

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
q3 = (int(visits.duplicated().sum()), int(visits.duplicated(subset="visit_id").sum()))
q3
```

```
(12, 20)
```

</details>

### Q4  Parsing visit_date

`visit_date` mixes `2025-03-14`, `14/03/2025` and `14-Mar-2025`, and some cells are blank.
Parse all three into datetimes, then count the visits that fall in March 2025.

Answer variable `q4`: an `int`.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
dates = pd.to_datetime(visits["visit_date"], format="mixed", dayfirst=True, errors="coerce")
q4 = int(((dates.dt.year == 2025) & (dates.dt.month == 3)).sum())
q4
```

```
173
```

</details>

### Q5  Mobile numbers from the phone column

`phone` holds `+91 98765 43210`, `9876543210`, `098765-43210` and `(044) 2345 6789`. Strip
everything that is not a digit and take the last ten. A first digit of 6, 7, 8 or 9 means a mobile.
Count those.

Answer variable `q5`: an `int`.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
last10 = visits["phone"].str.replace(r"\D", "", regex=True).str.extract(r"(\d{10})$")[0]
q5 = int(last10.str[0].isin(list("6789")).sum())
q5
```

```
533
```

</details>

### Q6  Kilograms and grams in weight_kg

`weight_kg` is in kilograms, except where somebody filed grams. Count the rows in grams.

Answer variable `q6`: an `int`.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
q6 = int((visits["weight_kg"] > 300).sum())
q6
```

```
77
```

</details>

### Q7  Outliers by the IQR rule

Repair `systolic_bp`, drop the whole-row duplicates, then flag the values outside
1.5 IQR of the quartiles. Count them.

Answer variable `q7`: an `int`.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
bp = pd.to_numeric(visits["systolic_bp"], errors="coerce").mask(lambda s: s == 999)
clean = visits.assign(systolic_bp=bp).drop_duplicates()

lo, hi = clean["systolic_bp"].quantile([0.25, 0.75])
iqr = hi - lo
q7 = int(((clean["systolic_bp"] < lo - 1.5 * iqr) | (clean["systolic_bp"] > hi + 1.5 * iqr)).sum())
q7
```

```
12
```

</details>

### Q8  Missingness patterns

Build a boolean frame over `visit_date`, `age` and `systolic_bp`, `True` where the value is
missing, counting `999` in `age` and the three sentinels in `systolic_bp`. Count the rows of each
pattern.

Answer variable `q8`: a Series indexed by the three booleans.

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
miss = pd.DataFrame({
    "visit_date": visits["visit_date"].isna(),
    "age": visits["age"].isna() | (visits["age"] == 999),
    "systolic_bp": visits["systolic_bp"].isna() | visits["systolic_bp"].isin(SENTINELS),
})
q8 = miss.value_counts()
q8
```

```
visit_date  age    systolic_bp
False       False  False          519
                   True            58
            True   False           19
True        False  False           17
False       True   True             3
True        True   False            3
            False  True             1
Name: count, dtype: int64
```

</details>

### Q9  Department means for imputation

Filling a missing `systolic_bp` from its department needs those means. Compute them on the
repaired, deduplicated frame, rounded to 1 decimal.

Answer variable `q9`: a Series indexed by department.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
bp = pd.to_numeric(visits["systolic_bp"], errors="coerce").mask(lambda s: s == 999)
clean = visits.assign(systolic_bp=bp).drop_duplicates()

q9 = clean.groupby("department")["systolic_bp"].mean().round(1)
q9
```

```
department
Cardiology          143.0
General Medicine    126.7
Orthopaedics        127.5
Paediatrics         108.5
Name: systolic_bp, dtype: float64
```

</details>

### Q10  A cleaning log row

Write the log row for the decision you made in Q6, as a dict with the keys `column`,
`decision`, `rows`, `assumption`, `why`. Wording is yours. Keep the five keys.

Answer variable `q10`: a `dict`.

In [ ]:
q10 = {
    "column": ...,
    "decision": ...,
    "rows": ...,
    "assumption": ...,
    "why": ...,
}

<details>
<summary><b>Answer</b></summary>

```python
from pprint import pprint

q10 = {
    "column": "weight_kg",
    "decision": "divided values above 300 by 1000",
    "rows": 77,
    "assumption": "no patient in this clinic weighs over 300 kg, so a large value is grams",
    "why": "the column mixes two units and the mean is meaningless until they agree",
}
pprint(q10, sort_dicts=False)
```

```
{'column': 'weight_kg',
 'decision': 'divided values above 300 by 1000',
 'rows': 77,
 'assumption': 'no patient in this clinic weighs over 300 kg, so a large value '
               'is grams',
 'why': 'the column mixes two units and the mean is meaningless until they '
        'agree'}
```

</details>